# Testing FraudLens with the real credit-card dataset

This notebook is the experiment worksheet. The actual training code lives in `src/models` and `scripts/benchmark_real_data.py` so the Kaggle run and local app use the same pipeline.

**Question:** How well do different classifiers identify rare fraudulent transactions, and what false-alarm tradeoff comes with the chosen model?

This notebook has not been executed as part of the repository update. Enter your own observed results after running it.


## 1. Set up the workspace

Attach Kaggle's `mlg-ulb/creditcardfraud` dataset. Copy or extract the **complete** FraudLens repository to `/kaggle/working/FraudLens`. If internet access is enabled, cloning the repository is another option; be sure your copy also contains the new benchmark script.

The project pins Python 3.12 dependencies. The notebook kernel may use another interpreter, so we create a separate 3.12 environment with `uv` in the following section.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = Path('/kaggle/working/FraudLens')
DATASETS = sorted(Path('/kaggle/input').rglob('creditcard.csv'))
assert (ROOT / 'src' / 'models' / 'train.py').is_file(), 'Copy the complete FraudLens repository into /kaggle/working/FraudLens first.'
assert (ROOT / 'scripts' / 'benchmark_real_data.py').is_file(), 'The benchmark script is missing.'
assert DATASETS, 'Attach the Credit Card Fraud Detection dataset to this notebook.'
DATA = DATASETS[0]
print('Dataset:', DATA)
print('Repository:', ROOT)
print('Notebook Python:', sys.version.split()[0])


## 2. Inspect the data

This is an exploratory check only, not a substitute for the validation module. Use aggregate statistics and look closely at the class balance.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(DATA)
print('Rows, columns:', df.shape)
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicated rows:', int(df.duplicated().sum()))
print('Class counts:')
print(df['Class'].value_counts().to_string())
print('Fraud share (%):', round(df['Class'].mean()*100, 4))
fig, axes = plt.subplots(1,2,figsize=(10,3.5))
df['Class'].value_counts().sort_index().plot.bar(ax=axes[0],title='Class counts')
axes[0].set_xlabel('Class (0 legitimate, 1 fraud)')
axes[0].set_ylabel('Transactions')
df['Amount'].clip(upper=df['Amount'].quantile(0.99)).hist(ax=axes[1],bins=40)
axes[1].set_title('Transaction amount (capped at 99th percentile)')
axes[1].set_xlabel('Amount')
plt.tight_layout()
plt.show()


## 3. Install the project dependencies in Python 3.12

Kaggle may require internet access for dependency installation. If internet is unavailable, use a prepared environment with the same pinned versions. This setup stays inside the temporary Kaggle session and does not alter your laptop.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
subprocess.run(['uv', 'venv', str(ROOT / '.venv'), '--python', '3.12'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(ROOT / '.venv' / 'bin' / 'python'), '-r', str(ROOT / 'requirements.txt')], check=True)
print('Project environment ready')


## 4. Run the benchmark using the real project pipeline

The script validates the source file, keeps a snapshot and checksum, trains five candidates, chooses the threshold on validation only, and reports metrics on the untouched chronological test set. It **does not** replace the active model by default.

Training may use significant memory or CPU time. No particular metric is expected in advance.


In [ ]:
python = ROOT / '.venv' / 'bin' / 'python'
env = dict(os.environ, FRAUDLENS_ROOT=str(ROOT))
result = subprocess.run([str(python), '-m', 'scripts.benchmark_real_data', '--input', str(DATA), '--split', 'chronological'], cwd=str(ROOT), env=env, check=True, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('Training messages:', result.stderr[-2500:])


## 5. Review the results before exporting anything

The release metadata contains the selected model, decision threshold and holdout scores. Write down errors, not only good scores. One high metric does not prove real-world fraud reduction.


In [ ]:
import json
releases = sorted((ROOT / 'models' / 'releases').glob('*/metadata.json'), key=lambda p:p.stat().st_mtime)
assert releases, 'No model release found. Check the benchmark log.'
metadata_path = releases[-1]
meta = json.loads(metadata_path.read_text())
print('Model:', meta['model_type'])
print('Split:', meta['split_strategy'])
print('Threshold:', meta['decision_threshold'])
print('Holdout rows:', meta['test_rows'], 'frauds:', meta['test_fraud'])
print('Holdout metrics:')
for k, v in meta['test_metrics'].items():
    print(f'  {k}: {v}')
print('Release:', metadata_path.parent)


## 6. Export a verified release

Download only the model release and its metadata, not the raw transaction CSV. Before deploying locally, install matching dependencies and check the artifact checksum. Do not deserialize untrusted joblib files. The separate application/database integration still needs acceptance testing.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
release = metadata_path.parent
export = Path('/kaggle/working/fraudlens-real-model-release.zip')
with ZipFile(export, 'w', compression=ZIP_DEFLATED) as archive:
    for file in release.rglob('*'):
        if file.is_file():
            archive.write(file, arcname=f'releases/{release.name}/{file.relative_to(release)}')
print('Exported:', export)
print('Remember to keep the source checksum and experiment notes with this release.')


## 7. Notes for the report

After running the notebook, record: the dataset checksum; valid/rejected row counts; split class counts; validation model ranking; champion threshold; test precision, recall, F1, average precision and ROC-AUC; confusion matrix; and observations about missed fraud and false alarms.

Do not claim the synthetic demonstration score as the result of this experiment. Keep any future tuning away from this final holdout.
